# DATALUS Generative Studio: Scenario Simulation & applied GenAI

This notebook demonstrates the **DATALUS Generative Studio**, focusing on applied Generative AI tasks using a pre-trained diffusion model. 
We explore how Data Scientists and Public Policy Makers can leverage synthetic data for scenario simulation, 
minority class balancing, tabular inpainting, and advanced data augmentation.

## 1. Environment Setup & Artifact Verification
Detecting environment and verifying pre-requisites. If training artifacts are missing, we run a lightning training pass.

In [ ]:
import os
import sys
from pathlib import Path
import polars as pl
import numpy as np

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_studio'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_studio'
    return './datalus_studio'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

# Install dependencies if in Colab/Kaggle
if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    !git clone https://github.com/emanuellcs/datalus.git || true
    %cd datalus
    !python -m pip install -e '.[dev]' pysus polars matplotlib seaborn
    sys.path.append(os.getcwd())

checkpoint_path = Path(f'{WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt')
if not checkpoint_path.exists():
    print('Training artifacts not found. Running lightning training pass...')
    from pysus.api import sih
    df = pl.from_pandas(sih(state='SP', year=2024, month=[1]))
    df = df.with_columns(pl.col('MORTE').cast(pl.Int64).fill_null(0)).head(5000)
    df.write_parquet(f'{WORKING_DIR}/raw_sample.parquet')
    
    !datalus ingest {WORKING_DIR}/raw_sample.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE
    !datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 1 --batch-size 256
else:
    print('Verified: Training artifacts are ready.')


## 2. Ab-Initio Generation & Distribution Analysis (KDE)
Generating a large synthetic dataset and comparing its statistical distribution to the real data using Kernel Density Estimation (KDE).

In [ ]:
!datalus sample {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet --n-records 10000 --cfg-scale 1.0

import matplotlib.pyplot as plt
import seaborn as sns

real = pl.read_parquet(f'{WORKING_DIR}/processed.parquet')
synth = pl.read_parquet(f'{WORKING_DIR}/synthetic.parquet')

plt.figure(figsize=(12, 6))
sns.kdeplot(data=real['IDADE'], label='Real Data', fill=True, alpha=0.3)
sns.kdeplot(data=synth['IDADE'], label='Synthetic Data', fill=True, alpha=0.3)
plt.title('Kernel Density Estimation: Real vs Synthetic (Age / IDADE)')
plt.xlabel('Age (Years)')
plt.ylabel('Density')
plt.legend()
plt.show()

## 3. Advanced Data Augmentation
Demonstrating how synthetic augmentation prevents overfitting in downstream machine learning tasks.

In [ ]:
seed = synth.sample(n=1000)
seed.write_parquet(f'{WORKING_DIR}/seed_augment.parquet')

!datalus augment {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/seed_augment.parquet {WORKING_DIR}/augmented.parquet --n-records 1000

## 4. Algorithmic Fairness: Minority Class Balancing
Addressing class imbalance (e.g., rare diseases or specific demographics) through oversampling without row duplication.

In [ ]:
!datalus balance {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/balanced.parquet MORTE '{"1": 5000}'

balanced = pl.read_parquet(f'{WORKING_DIR}/balanced.parquet')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
real['MORTE'].value_counts().to_pandas().plot(kind='bar', x='MORTE', y='count', ax=ax1, color='salmon')
ax1.set_title('Original Class Imbalance (Real)')
balanced['MORTE'].value_counts().to_pandas().plot(kind='bar', x='MORTE', y='count', ax=ax2, color='skyblue')
ax2.set_title('Balanced Class Distribution (Synthetic)')
plt.show()

## 5. Tabular Inpainting (Imputation)
Using RePaint-style inference to conditionally fill null values while preserving observed fields.

In [ ]:
incomplete = real.head(100).with_columns(
    pl.when(pl.Series(np.random.rand(100) > 0.5)).then(None).otherwise(pl.col('IDADE')).alias('IDADE')
)
incomplete.write_parquet(f'{WORKING_DIR}/incomplete.parquet')

!datalus inpaint {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/incomplete.parquet {WORKING_DIR}/inpainted.parquet

## 6. Counterfactual Interventions & Policy Simulation
Simulating "What if" scenarios by modifying patient traits and allowing the model to regenerate the clinical record coherently.

In [ ]:
!datalus counterfactual {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/synthetic.parquet {WORKING_DIR}/counterfactual.parquet '{"SEXO": "1"}'